# Notebook 05 — Individual Model Comparison (9 configurations)

**Paper artifact:** Table *"Performance comparison of all YOLO26 configurations on the Target Domain"* (`tab:model_metrics_summary`).

This notebook evaluates the nine trained YOLO26 detectors — the Cartesian product of three **backbone sizes** (Nano, Medium, Large) and three **training protocols** (Aerial-only, Underwater / Direct Transfer, Mixed / Sequential Transfer) — on the held-out underwater test partition (SINTEF LIACI) and consolidates Precision, Recall, F1-Score, mAP@50 and mAP@[.5:.95] into a single comparison table.

**Key result.** The *Underwater–Medium* configuration (Direct Transfer) dominates every metric; the *Mixed* (Sequential Transfer) and *Aerial-only* protocols collapse, empirically confirming the negative-transfer hypothesis that motivates the paper's Direct Transfer design.

---

| | |
|---|---|
| **Inputs** | nine `.pt` checkpoints in `modelos_entrenados/`; `dataset_yolo/dataset.yml` (test split) |
| **Output** | consolidated metrics table (Section 4) → paper `tab:model_metrics_summary` |
| **Environment** | Ultralytics 8.4.5 · PyTorch 2.8.0+cu128 · Python 3.9.13 · NVIDIA RTX 4090 |

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed on the original GPU run; Ultralytics console logs appear in their original form. Re-running under `requirements.txt` reproduces the same table.

## 1 · Environment and GPU setup

Import the inference stack (Ultralytics + PyTorch) and reset the CUDA cache to establish a clean memory baseline before the evaluation sweep.

In [1]:
import os
import pandas as pd
from ultralytics import YOLO
import torch

def clear_gpu_cache():
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        print(f"GPU cache cleared on device {device}.")
    else:
        print("CUDA is not available.")
clear_gpu_cache()

GPU cache cleared on device 0.


## 2 · Model zoo and dataset paths

We register the nine checkpoints — three backbones (Nano / Medium / Large) × three training protocols — and point to the underwater evaluation set.

**Naming convention:** `Aereo` = Aerial-only · `Acuatico` = Underwater (Direct Transfer) · `Mixto` = Mixed (Sequential Transfer aerial→underwater).

In [2]:
# --- 1. Path configuration ---

# Auto-detect the trained-models directory
if os.path.exists('entrenados'):
    MODELS_DIR = 'entrenados'
elif os.path.exists('modelos_entrenados'):
    MODELS_DIR = 'modelos_entrenados'
else:
    MODELS_DIR = '' # Not found

ACUATICO_YAML = './dataset_yolo/dataset.yml'

models_to_evaluate = {
    'Aerial-Nano': 'modelo-aereo-n.pt',
    'Aerial-Medium': 'modelo-aereo-m.pt',
    'Aerial-Large': 'modelo-aereo-l.pt',
    'Underwater-Nano': 'modelo-acuatico-n.pt',
    'Underwater-Medium': 'modelo-acuatico-m.pt',
    'Underwater-Large': 'modelo-acuatico-l.pt',
    'Mixed-Nano': 'modelo-mixto-n.pt',
    'Mixed-Medium': 'modelo-mixto-m.pt',
    'Mixed-Large': 'modelo-mixto-l.pt',
}

# Sanity checks
if not MODELS_DIR:
    print("WARNING: neither 'entrenados' nor 'modelos_entrenados' was found. Cannot continue.")
else:
    print(f"Models directory: '{MODELS_DIR}'")
print('Path configuration complete.')

Models directory: 'modelos_entrenados'
Path configuration complete.


## 3 · Evaluation routine

Each checkpoint is evaluated with `model.val(split='test')` on the **held-out** test partition (33 images, 52 instances) that was never used for training or threshold selection. Ultralytics reports Precision, Recall, F1, mAP@50 and mAP@[.5:.95] at its standard operating point — identical settings for all nine models, ensuring a fair comparison.

In [3]:
def run_evaluation(dataset_yaml, dataset_name):
    """Evaluate every registered model on a given dataset split."""
    results = {}
    print(f"\n--- STARTING EVALUATION ON DATASET {dataset_name.upper()} ---")
    if not MODELS_DIR:
        print("Cannot evaluate: trained-models directory not found.")
        return results

    for name, model_file in models_to_evaluate.items():
        path = os.path.join(MODELS_DIR, model_file)
        if not os.path.exists(path):
            print(f"\nWARNING: model not found: {name} en {path} . Skipping.")
            results[name] = None
            continue
        try:
            print(f"\nEvaluating: {name}...")
            model = YOLO(path)
            metrics = model.val(data=dataset_yaml, split='test', name=f'eval_{name}_on_{dataset_name.lower()}')
            results[name] = metrics
        except Exception as e:
            print(f"\nERROR while evaluating model {name}: {e}")
            results[name] = None
    print(f"\n--- EVALUATION ON {dataset_name.upper()} COMPLETE ---")
    return results

# Run the evaluation
results_acuatico = run_evaluation(ACUATICO_YAML, 'Acuatico')


--- STARTING EVALUATION ON DATASET ACUATICO ---

Evaluating: Aerial-Nano...
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2786.6±772.2 MB/s, size: 28.9 KB)
val: Scanning /home/user/work/EONSEA/Articulo-Corrosion/dataset_yolo/labels/test... 21 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 33/33 2.1Kit/s 0.0s
WARNING ⚠️ val: Cache directory /home/user/work/EONSEA/Articulo-Corrosion/dataset_yolo/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 7.3it/s 0.4s0.3s
                   all         33         52      0.163     0.0769     0.0392     0.0163
Speed: 2.7ms preprocess, 4.1ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /home/user/work/EONSEA/Articulo-Corrosion/runs/detect/ev

## 4 · Consolidated results table

The per-model metrics are gathered into a single `DataFrame`, sorted by mAP@50. **This table is the source of truth for the paper's nine-row model comparison.**

In [4]:
def create_and_print_df(results_dict, dataset_name):
    """Build and print a tidy results DataFrame."""
    print(f"\n--- RESULTS TABLE: DATASET {dataset_name.upper()} ---")
    data_for_df = []
    for name, metrics in results_dict.items():
        if metrics and hasattr(metrics, 'box'):
            data_for_df.append({
                'Model': name,
                'mAP50-95': metrics.box.map,
                'mAP50': metrics.box.map50,
                'Precision': metrics.box.p[0],
                'Recall': metrics.box.r[0],
                'F1-Score': metrics.box.f1[0]
            })
        else:
            # Include a row for models that were missing or failed
            data_for_df.append({'Model': name, 'mAP50-95': 0, 'mAP50': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0})

    if not data_for_df:
        print("No results were generated.")
        return

    df = pd.DataFrame(data_for_df).sort_values(by='mAP50', ascending=False).reset_index(drop=True)
    print(df.to_string(float_format='{:.4f}'.format))

# Display the results table
create_and_print_df(results_acuatico, 'Acuatico')


--- RESULTS TABLE: DATASET ACUATICO ---
            Model  mAP50-95  mAP50  Precision  Recall  F1-Score
0  Underwater-Medium    0.4884 0.8196     0.8084  0.7308    0.7676
1   Underwater-Large    0.3548 0.6706     0.5737  0.6987    0.6300
2    Underwater-Nano    0.2265 0.4938     0.6285  0.4615    0.5322
3      Mixed-Large    0.0913 0.2259     0.5217  0.1923    0.2810
4       Mixed-Nano    0.0672 0.1548     0.3320  0.2308    0.2723
5     Mixed-Medium    0.0708 0.1275     0.4519  0.1346    0.2074
6       Aerial-Nano    0.0163 0.0392     0.1627  0.0769    0.1045
7     Aerial-Medium    0.0099 0.0188     0.1745  0.0577    0.0867
8      Aerial-Large    0.0121 0.0183     0.1682  0.0192    0.0345


### Reading the table → paper `tab:model_metrics_summary`

- **Underwater–Medium** is the strongest detector on every metric (F1 0.7676, mAP@50 0.8196). It is the backbone carried forward to the WBF ensemble (Notebook 06) and the TTA experiments (Notebook 07).
- **Scaling hierarchy:** Medium > Large > Nano under Direct Transfer. Nano lacks the capacity to model multi-scale corrosion, while Large overfits the small (N = 261) underwater set — see the overfitting analysis figure in the paper.
- **Negative transfer:** both the *Mixed* (Sequential) and *Aerial-only* protocols degrade sharply (F1 ≤ 0.28), confirming that aerial corrosion features are distributionally antagonistic to the underwater domain. This is the empirical motivation for the Direct Transfer protocol.

These nine rows are reproduced verbatim in the paper's model-comparison table as **point estimates** (no cross-fold ±std is claimed).